# **Import des modules nécessaires**

In [21]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic
from sklearn.cluster import KMeans #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [55]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",214
1,1865 La confession de Claude.,1865,1,2,"La mansarde entière me réclame les rires, les ...",187
2,1865 La confession de Claude.,1865,1,3,Le grillon chantait; le souffle harmonieux des...,146
3,1865 La confession de Claude.,1865,1,4,"brunes et rieuses filles, étaient reines des m...",190
4,1865 La confession de Claude.,1865,1,5,"Pars cependant, puisque tu as soif de la vie. ...",126


In [56]:
df.shape

(21260, 6)

# **Traitement du Corpus de Zola**

In [57]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

In [58]:
# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=50): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver matin frais manteau brouillard saison so...
1,mansarde entier rire richesse sœur foyer feu j...
2,grillon souffle harmonieux causerie lèvre cœur...
3,brun rieur moisson vendange épi grappe sentier...
4,soif projet soi ferme loyal action rêve vis gr...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [59]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [60]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/665 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

In [74]:
hdbscan_model = HDBSCAN( min_cluster_size=25, min_samples=2, metric='euclidean', cluster_selection_method='eom',prediction_data=True)

umap_model = UMAP( n_neighbors=20,n_components=3, min_dist=0.0, metric="cosine",random_state=42)

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [75]:
vectorizer_model = CountVectorizer(
    min_df=2,    # Le mot doit apparaître dans au moins 2 segments pour être pris en compte (élimine les fautes ou mots uniques)
    max_df=0.6)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

2026-07-07 17:02:49,097 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-07 17:02:58,797 - BERTopic - Dimensionality - Completed ✓
2026-07-07 17:02:58,798 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-07 17:03:00,363 - BERTopic - Cluster - Completed ✓
2026-07-07 17:03:00,366 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-07 17:03:00,625 - BERTopic - Representation - Completed ✓


In [76]:
new_topics = topic_model.reduce_outliers(
    df["phrases_lemm"].tolist(), 
    topics, 
    strategy="embeddings",
    embeddings=embeddings
)

# Met à jour le modèle avec ces nouveaux thèmes plus propres
topic_model.update_topics(df["phrases_lemm"].tolist(), topics=new_topics)

2026-07-07 17:03:02,576 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


## **3) Topics Présent**

In [77]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,640,0_amour_cœur_pensée_amant,"[amour, cœur, pensée, amant, passion, chair, t...",[fatalité physiologique corps action volonté t...
1,1,560,1_affaire_franc_fortune_instituteur,"[affaire, franc, fortune, instituteur, fils, f...",[vicomte vice-président franc prime secret exa...
2,2,339,2_cher_dame_mari_joli,"[cher, dame, mari, joli, ami, franc, idée, aff...",[fier continuel visite évêque prince église ca...
3,3,331,3_bonheur_raison_écoute_amour,"[bonheur, raison, écoute, amour, heureux, larm...",[arbre poison bête taillis noir venin vipère r...
4,4,277,4_horizon_arbre_ciel_soleil,"[horizon, arbre, ciel, soleil, gauche, vert, b...",[gauche toit lointain gris bleu brume ciel dro...
...,...,...,...,...,...
157,157,141,157_rire_bouquet_joie_foule,"[rire, bouquet, joie, foule, pelouse, rose, ta...",[appartement princier long rampe tapis haut la...
158,158,99,158_prêtre_pape_livre_beauté,"[prêtre, pape, livre, beauté, doute, palais, s...",[bête orgueil entêtement maison prospère jeune...
159,159,58,159_million_capital_société_argent,"[million, capital, société, argent, action, ra...",[étonnement hostile plaisanterie facile méchan...
160,160,104,160_larme_cri_malheureux_mal,"[larme, cri, malheureux, mal, sanglot, colère,...",[côté viande doigt allumette ténèbre cadavre c...


In [73]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

themes_interet = [1, 3, 7, 10, 11, 14]

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=20 
)

# Visualisation interactive de l'évolution des 10 premiers topics
topic_model.visualize_topics_over_time(topics_over_time, topics=themes_interet)

19it [00:06,  3.02it/s]


In [33]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('femme', np.float64(0.029430996301289526)), ('bon', np.float64(0.027966058982263557)), ('jeune', np.float64(0.02649416114109179)), ('air', np.float64(0.025098217426595654)), ('voix', np.float64(0.02393076483801953)), ('homme', np.float64(0.023923437147664924)), ('architecte', np.float64(0.023775786621955997)), ('dame', np.float64(0.02316554308976046)), ('jour', np.float64(0.020942349018651395)), ('monsieur', np.float64(0.020599738652537936))]

TOPIC 1
[('monsieur', np.float64(0.05343878731420725)), ('franc', np.float64(0.04022359284255543)), ('femme', np.float64(0.038389517695313125)), ('argent', np.float64(0.034466181587445036)), ('fille', np.float64(0.03363920505802684)), ('homme', np.float64(0.028036949433943355)), ('père', np.float64(0.0253491477534095)), ('mari', np.float64(0.023299568900134106)), ('vou', np.float64(0.02296642161442363)), ('bon', np.float64(0.02246071775882005))]

TOPIC 2
[('oncle', np.float64(0.02647236140573398)), ('femme', np.float64(0.02317440788185